In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from sqlalchemy import Column, FLOAT, Integer, MetaData, select, String, Table, create_engine

from llama_index.core import PromptTemplate, SQLDatabase, VectorStoreIndex
from llama_index.core.objects import ObjectIndex, SQLTableSchema
from llama_index.core.query_engine import NLSQLTableQueryEngine, SQLTableRetrieverQueryEngine
from llama_index.core.schema import TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LlamaOpenAI

# ==========================================
# OMGEVING & CONFIGURATIE
# ==========================================

# Laad de omgevingsvariabelen in vanuit het .env bestand
load_dotenv(dotenv_path="./../.env", override=True)

# Controleer of de API-sleutel aanwezig is
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(" OPENAI_API_KEY niet gevonden! Controleer je .env bestand.")

# LLM initialiseren (modelnaam behouden zoals in origineel)
llm = LlamaOpenAI(temperature=0.0, model="gpt-4.1-nano")

In [ ]:
# ==========================================
# DATABASE & TABEL DEFINITIE
# ==========================================

# Engine aanmaken met het juiste pad
engine = create_engine("sqlite:///../data/voeding.db")
metadata_obj = MetaData()

# De SQL-tabel voor voeding definiëren
table_name = "Food"
food_table = Table(
    table_name,
    metadata_obj,
    Column("NEVO-code", Integer(), primary_key=True),
    Column("productnaam", String(), nullable=False),
    Column("energie_kcal", FLOAT()),
    Column("koolhydraten_g", FLOAT()),
    Column("eiwitten_g", FLOAT()),
    Column("vetten_g", FLOAT()),
)

# Tabel aanmaken in de database
metadata_obj.create_all(engine)

In [ ]:
# ==========================================
# DATA INLADEN VANUIT CSV
# ==========================================

# Koppel de database aan LlamaIndex
sql_database = SQLDatabase(engine, include_tables=["Food"])

# CSV inladen in een Pandas DataFrame
df_import = pd.read_csv(
    "../data/food/NEVO_cleaned.csv"
)

# DataFrame wegschrijven naar de SQLite database
df_import.to_sql(name="Food", con=engine, if_exists="replace", index=False)

In [ ]:
# ==========================================
# DATABASE VALIDATIE & BASIS NL-SQL TEST
# ==========================================

# Test of de data goed in de database zit
with engine.connect() as conn:
    result = conn.execute(select(food_table).limit(3))
    for row in result:
        print(row)

# Basis Query Engine opzetten voor natuurlijke taal naar SQL
nl_query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database, tables=["Food"], llm=llm
)

# Testvraag uitvoeren
query_str = "welk product heeft de meeste eiwitten"
response = nl_query_engine.query(query_str)
display(Markdown(f"<b>{response}</b>"))

In [ ]:
# ==========================================
# 1. ROW RETRIEVAL (Vector Index voor rijen)
# ==========================================

# Haal alle productnamen op uit de database
with engine.connect() as connection:
    # We halen specifiek de 'productnaam' kolom op
    results = connection.execute(select(food_table.c.productnaam)).fetchall()

# Pak index [0] van elke rij (aangezien fetchall tuples teruggeeft, bijv. ('Andijvie gekookt',))
food_nodes = [TextNode(text=str(t[0])) for t in results]

# Bouw de vector index op voor deze productnamen
food_rows_index = VectorStoreIndex(
    food_nodes, embed_model=OpenAIEmbedding(model="text-embedding-3-small")
)

# Sla de index lokaal op
persist_dir = "../data/vectorstore_voeding_rijen"
food_rows_index.storage_context.persist(persist_dir=persist_dir)
print(" Vector Index voor productnamen succesvol opgeslagen op schijf!")

# Maak de retriever aan (top_k op 3 voor iets meer context-speling)
food_rows_retriever = food_rows_index.as_retriever(similarity_top_k=3)

# Stop de retriever in een dictionary, gekoppeld aan de exacte tabelnaam
rows_retrievers = {
    "Food": food_rows_retriever,
}

In [ ]:
# ==========================================
# 2. TABLE RETRIEVAL (Object Index)
# ==========================================
# Omdat we SQLTableRetrieverQueryEngine gebruiken, MOETEN we een tabel-retriever meegeven.
# Dit vertelt het systeem welke tabellen er überhaupt bestaan en wat er in staat.

# Een schone, simpele instructie met de exacte kolomnamen
table_instruction = (
    "Deze tabel bevat voedingswaarden van producten. "
    "De beschikbare kolommen zijn: 'NEVO-code', 'productnaam', 'energie_kcal', "
    "'koolhydraten_g', 'eiwitten_g', en 'vetten_g'. "
    "Gebruik uitsluitend deze exacte kolomnamen wanneer je een SQL query genereert."
)

my_sql_prompt_str = """
Je bent een behulpzame AI-voedingsexpert. Jouw taak is om een syntactisch correcte SQLite query te schrijven om de vraag van de gebruiker te beantwoorden.
Volg deze regels:
1. Gebruik UITSLUITEND de tabellen en kolommen die in de schema-informatie staan.
2. Let goed op de geleverde rij-hints (Row Retrieval) voor de exacte spelling van de productnaam.
3. Geef het uiteindelijke antwoord altijd in vloeiend Nederlands.
4. Als de voedingswaarde '0' is, zeg dan expliciet dat het product dit niet bevat.

Schema informatie:
{schema}

Vraag van de gebruiker: {query_str}
SQLQuery: 
"""
my_sql_prompt = PromptTemplate(my_sql_prompt_str)

# Bouw het schema-object op voor de tabel
table_schema_objs = [
    SQLTableSchema(
        table_name="Food",
        context_str=table_instruction
    )
]

# Bouw de Object Index op voor de tabellen
obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    index_cls=VectorStoreIndex,
)

In [ ]:
# ==========================================
# 3. DE COMBINATIE (SQLTableRetrieverQueryEngine) & TESTEN
# ==========================================

query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    # Haalt de beste tabel op (we hebben er maar 1, dus top_k=1 is perfect)
    obj_index.as_retriever(similarity_top_k=1),
    # Haalt de beste rijen op uit die tabel als hints/context
    rows_retrievers=rows_retrievers,
    # text_to_sql_prompt=my_sql_prompt, # Optioneel in te schakelen
    # synthesize_response=False geeft enkel de opgehaalde (ruwe) gegevens terug in plaats van een samengestelde tekst
    synthesize_response=False
)

# --- Testen ---
response = query_engine.query("Hoeveel eiwitten en koolhydraten zitten er in gekookte andijvie?")
print(response)